# Data Cleaning

This notebook handles data preparation for the weather forecasting project.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/GlobalWeatherRepository.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (138193, 41)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 138193 entries, 0 to 138192
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   country                       138193 non-null  str    
 1   location_name                 138193 non-null  str    
 2   latitude                      138193 non-null  float64
 3   longitude                     138193 non-null  float64
 4   timezone                      138193 non-null  str    
 5   last_updated_epoch            138193 non-null  int64  
 6   last_updated                  138193 non-null  str    
 7   temperature_celsius           138193 non-null  float64
 8   temperature_fahrenheit        138193 non-null  float64
 9   condition_text                138193 non-null  str    
 10  wind_mph                      138193 non-null  float64
 11  wind_kph                      138193 non-null  float64
 12  wind_degree                   138193 non-null  int64  


In [3]:
df.isnull().sum()

country                         0
location_name                   0
latitude                        0
longitude                       0
timezone                        0
last_updated_epoch              0
last_updated                    0
temperature_celsius             0
temperature_fahrenheit          0
condition_text                  0
wind_mph                        0
wind_kph                        0
wind_degree                     0
wind_direction                  0
pressure_mb                     0
pressure_in                     0
precip_mm                       0
precip_in                       0
humidity                        0
cloud                           0
feels_like_celsius              0
feels_like_fahrenheit           0
visibility_km                   0
visibility_miles                0
uv_index                        0
gust_mph                        0
gust_kph                        0
air_quality_Carbon_Monoxide     0
air_quality_Ozone               0
air_quality_Ni

## Handling Missing Values

In [4]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=['object']).columns

for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in categorical_cols:
    mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
    df[col] = df[col].fillna(mode_val)

print(f'Missing values after imputation: {df.isnull().sum().sum()}')

C:\Users\theot\AppData\Local\Temp\ipykernel_1488\4154478485.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


Missing values after imputation: 0


## Converting Timestamps

In [5]:
df['lastupdated'] = pd.to_datetime(df['last_updated'])

# Remove duplicate rows (same location + same timestamp)
df = df.drop_duplicates(subset=['location_name', 'lastupdated'])
print(f'Rows after deduplication: {len(df)}')

# Sort by location, then by time (critical for correct lag features)
df = df.sort_values(['location_name', 'lastupdated']).reset_index(drop=True)
print(f'Date range: {df["lastupdated"].min()} to {df["lastupdated"].max()}')
print(f'Locations: {df["location_name"].nunique()}')

Rows after deduplication: 138192
Date range: 2024-05-16 01:45:00 to 2026-04-28 19:30:00
Locations: 257


## Removing Outliers (IQR Method)

In [6]:
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

clean_df = df.copy()
for col in ['temperature_celsius', 'humidity', 'precip_mm']:
    if col in clean_df.columns:
        clean_df = remove_outliers_iqr(clean_df, col)

print(f'Rows after outlier removal: {len(clean_df)}')

Rows after outlier removal: 110486


In [7]:
# Rename columns to standard names
clean_df = clean_df.rename(columns={
    'temperature_celsius': 'temperature',
    'precip_mm': 'precipitation'
})

clean_df.to_csv('../data/cleaned_weather.csv', index=False)
print('Saved cleaned data to ../data/cleaned_weather.csv')
print(f'Columns: {list(clean_df.columns)}')

Saved cleaned data to ../data/cleaned_weather.csv
Columns: ['country', 'location_name', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'last_updated', 'temperature', 'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph', 'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in', 'precipitation', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination', 'lastupdated']


## Summary

**What was done:**
- Filled missing numeric values with median (robust to outliers)
- Filled missing categorical values with mode
- Converted lastupdated to datetime and sorted chronologically
- Removed extreme outliers using IQR method

**Why these methods:**
- Median is better than mean for weather data which can have extreme values
- IQR method is simple and effective for removing clear outliers without losing too much data
- Time-based sorting is essential for time series forecasting